In [ ]:
!pip install confluent-kafka

In [ ]:
from confluent_kafka import Consumer, KafkaException, KafkaError
import psycopg2
import json

In [ ]:
import os

config = {
    'bootstrap.servers': os.environ['KAFKA_BOOTSTRAP_SERVERS'],
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': os.environ['KAFKA_API_KEY'],
    'sasl.password': os.environ['KAFKA_API_SECRET'],
    'group.id': os.getenv('KAFKA_GROUP_ID', 'analytics-consumer'),
    'auto.offset.reset': 'earliest',
}


In [ ]:
consumer = Consumer(**config)


In [ ]:
# Subscribe to the 'users' topic
topic = 'topic_0'
consumer.subscribe([topic])#

In [ ]:
import os

postgres_conf = {
    'dbname': os.getenv('AIVEN_DB_NAME', 'defaultdb'),
    'user': os.environ['AIVEN_DB_USER'],
    'password': os.environ['AIVEN_DB_PASSWORD'],
    'host': os.environ['AIVEN_DB_HOST'],
    'port': os.getenv('AIVEN_DB_PORT', '17508'),
    'sslmode': 'require',
}


In [ ]:
conn = psycopg2.connect(**postgres_conf)
cursor = conn.cursor()

# create_table = """
# CREATE TABLE tbl_messages (
#     id SERIAL PRIMARY KEY,
#     Open Varchar(255),
#     High Varchar(255),
#     Low Varchar(255),
#     Close Varchar(255),
#     Volume Varchar(255)
# );
# # """
# cursor.execute(create_table)
# conn.commit()

In [ ]:
try:
    while True:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                # End of partition event
                print('%% %s [%d] reached end at offset %d\n' %
                      (msg.topic(), msg.partition(), msg.offset()))
            elif msg.error():
                raise KafkaException(msg.error())
        else:
            # Print the message key and
            print('Received message: key=%s value=%s' % (msg.key().decode('utf-8'), msg.value().decode('utf-8')))
except KeyboardInterrupt:
    pass
finally:
    # Close down consumer to commit final offsets.
    consumer.close()

In [ ]:
def insert_data(data):
  values = '''
  insert into tbl_messages (Open, High, Low, Close, Volume)
  values (%s, %s, %s, %s, %s)

  '''
  cursor.execute(values, (data['1. open'], data[ '2. high'], data['3. low'], data['4. close'], data['5. volume']))
  conn.commit()
  print("Data inserted successfully")



In [ ]:
consumer = Consumer(**config)
topic = 'topic_0'
consumer.subscribe([topic])

In [ ]:
conn = psycopg2.connect(**postgres_conf)
cursor = conn.cursor()
# Consume messages
try:
    while True:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                # End of partition event
                print('%% %s [%d] reached end at offset %d\n' %
                      (msg.topic(), msg.partition(), msg.offset()))
            elif msg.error():
                raise KafkaException(msg.error())
        else:
            # Print the message key and value
            print('Received message: key=%s value=%s' % (msg.key().decode('utf-8') if msg.key() else None, msg.value().decode('utf-8') if msg.value() else None))
            data = msg.value().decode('utf-8')
            msg = json.loads(data)

            # Insert data into PostgreSQL
            insert_data(msg)

except KeyboardInterrupt:
    pass
finally:
    # Close down consumer to commit final offsets.
    consumer.close()
    # Close PostgreSQL connection
    cursor.close()
    conn.close()